## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [ ]:
import os
from dotenv import load_dotenv
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [14]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Capital of India")
print(response.content)


<think>
Okay, so the user asked, "Capital of India." Let me think about how to approach this. First, I need to confirm the correct answer. I remember that New Delhi is the capital, but I should make sure there's no confusion with other cities like Mumbai or Kolkata.

Wait, Mumbai is the financial capital, right? And Kolkata was the capital during British rule, but that changed to Delhi later. So the current capital is definitely New Delhi. I should mention that it's in the National Capital Territory of Delhi. Also, maybe some people confuse Delhi and New Delhi, so clarifying that would be helpful. Let me check if there's any other possible capitals or if there's a historical aspect to include. Oh, right, during British India, the capital was Kolkata, then shifted to Delhi in 1911. That's probably useful context. I should also note that while New Delhi is the capital, Delhi as a whole is a union territory. Let me structure this clearly: start with the direct answer, then provide context

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location : str) -> str:
    """Get the weather at a location"""
    return "Its Sunny in {location} :)"

model_with_tools = model.bind_tools([get_weather])


_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023A9E8FAD70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023A9E8F95B0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [19]:
response = model_with_tools.invoke("Get weather of Hyderabad")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Tool: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Hyderabad. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Hyderabad is the location they mentioned. So I\'ll call the function with "Hyderabad" as the location. Make sure the JSON is correctly formatted with the name and arguments. Let me double-check the syntax. Everything looks good. Time to send the tool call.\n', 'tool_calls': [{'id': '91sf7m6bj', 'function': {'arguments': '{"location":"Hyderabad"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 150, 'total_tokens': 268, 'completion_time': 0.209772114, 'completion_tokens_details': {'reasoning_tokens': 93}, 'prompt_time': 0.007259412, 'prompt_tokens_details': None, 'queue_time': 0.050704428, 'total_time': 0.217031526}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint'

### Tool Execution loop

In [39]:
# Step 1: Model generates tool calls
messages = [{"role":"user", "content": "What is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
print("AI Message ----> ", ai_msg)
messages.append(ai_msg)
print("Meassage+AI Message ----> ", messages)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    print("tool_result ----> ", tool_result)
    messages.append(tool_result)
print("Meassage+AI Message+Tool message ----> ", messages)    

print("****"*10)
# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."


AI Message ---->  content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. Let me format the tool call correctly.\n', 'tool_calls': [{'id': 'm7ze25avq', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 153, 'total_tokens': 245, 'completion_time': 0.152312004, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.007256229, 'prompt_tokens_details': None, 'queue_time': 0.049691271, 'total_time': 0.159568233}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': '